# Example : control loop

In this example we explain how to us the control loop sub package of freegsnke. We will demonstrate the individual features and then look at running a validation test and a full dynamic simulation.

First we will follow the usual first steps of creating an tokamak, GS equilibrium and profiles. Then we will 
- create a sample control schedule, waveforms and VC schedule.
- demonstrate the use of the controller objects for Ip control an shape control
- run a validation test using the full control system
- run a dynamic simulation using the full control system



## import packages 

In [ ]:
import numpy as np
import pickle
import time
import os
import matplotlib.pyplot as plt
from copy import deepcopy

import freegs4e
from freegsnke import equilibrium_update, GSstaticsolver

%load_ext autoreload
%autoreload 2

## Create Machine and equilibrium

- set paths to config files
- create tokamak object
- create equilibrium object
- solve equilibrium

In [ ]:
# set paths
os.environ["ACTIVE_COILS_PATH"] = (
    f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle"
)
os.environ["PASSIVE_COILS_PATH"] = (
    f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle"
)
os.environ["WALL_PATH"] = f"../machine_configs/MAST-U/MAST-U_like_wall.pickle"
os.environ["LIMITER_PATH"] = f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle"

In [ ]:
# Now the machine can actually be built:
from freegsnke import build_machine

tokamak = build_machine.tokamak()

### create blank equilibrium and profiles

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=2.0,  # Radial range
    Zmin=-2.2,
    Zmax=2.2,  # Vertical range
    nx=65,  # Number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,  # Number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)

In [ ]:
from freegsnke.jtor_update import ConstrainPaxisIp

profiles = ConstrainPaxisIp(
    eq=eq,  # equilibrium object
    paxis=8.1e3,  # profile object
    Ip=6.2e5,  # plasma current
    fvac=0.5,  # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2,  # profile function parameter
)

In [ ]:
eq_copy = deepcopy(eq)
profiles_copy = deepcopy(profiles)

### set coil currents and solve equilibrium

We'll use the same sample currents as in previous examples.

In [ ]:
with open("simple_diverted_currents_PaxisIp.pk", "rb") as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak[key].current = current_values[key]

# instantiate solver
from freegsnke import GSstaticsolver

GSStaticSolver = GSstaticsolver.NKGSsolver(eq)

# Call solver and solve for equilibrium
GSStaticSolver.solve(
    eq=eq, profiles=profiles, constrain=None, target_relative_tolerance=1e-8, verbose=0
)

we can plot this equilibrium

In [ ]:
from freegs4e.plotting import plotEquilibrium as plot_eqi

plot_eqi(eq, show=True)

The simulations will also make use of the non linear solver so we'll create that now here too.



In [ ]:
from freegsnke.nonlinear_solve import nl_solver

stepping = nl_solver(
    eq=eq,
    profiles=profiles,
    full_timestep=5e-4,
    plasma_resistivity=1e-6,
    min_dIy_dI=0.1,
    max_mode_frequency=10**2.5,
)

# set initial conditions
stepping.initialize_from_ICs(eq, profiles)

## Creating a control schedule

We need to specify how we'd like the plasma to evolve in time. This is done by setting schedules and waveforms for the plasma current and shape targets. As a simplest example we'll set these to remain constant throughout the simulation. This data is stored as a dictionary and saved to pickle file. 

The schedule prescribes which quantites are to be controlled at which time. We need a schedule for the feedfoward and feedback shape control, and for Ip control.

The waveforms prescribe the values of the quantities at which the schedules are to be controlled. We also need waveforms for the feedforward and feedback shape control, and for Ip control. We'll use the same waveform for feedback and feedfoward (usually these will be the same)

We'll create two schedules for each, one where there is no control and one where there is, and we'll use the same waveform for both.

### Shape targets 

For the shape control we'll use the following targets:

[R_in, R_out, Rx_lower, Rs_lower_outer]



In [ ]:
# blank schedule - target control is turned off
targ_schedule_1 = {
    0.0: [],
}

# schedule with target control on
targ_schedule_2 = {
    0.0: ["R_in", "R_out", "Rx_lower"],
    0.2: ["R_in", "R_out", "Rx_lower", "Rs_lower_outer"],
}

# save to pickle

with open("./control_pickles/shape_schedule_off.pkl", "wb") as f:
    pickle.dump(targ_schedule_1, f)

with open("./control_pickles/shape_schedule_on.pkl", "wb") as f:
    pickle.dump(targ_schedule_2, f)

To create the waveform we'll start at the shape values of our start equilibrium. If we chose values too far away then the dynamic solver will not converge. We can use the VirtualCirucitHandling class to calculate the shape values of the equilibrium, as well as the VC itself. 

We'll need to instantiate the VirtualCirucitHandling class, and then call the calculate targets


In [ ]:
# waveform for target control.
from freegsnke.virtual_circuits import VirtualCircuitHandling

VCH = VirtualCircuitHandling()
# assign solver to VCH
VCH.define_solver(solver=GSStaticSolver, target_relative_tolerance=1e-7)
targets_names, start_target_vals = VCH.calculate_targets(
    eq, targets=["R_in", "R_out", "Rx_lower", "Rs_lower_outer"]
)

shape_waveform_const = {target: {} for target in targets_names}
shape_waveform_const["R_in"]["times"] = [0.0]
shape_waveform_const["R_in"]["vals"] = [start_target_vals[0]]
shape_waveform_const["R_out"]["times"] = [0.0]
shape_waveform_const["R_out"]["vals"] = [start_target_vals[1]]
shape_waveform_const["Rx_lower"]["times"] = [0.0]
shape_waveform_const["Rx_lower"]["vals"] = [start_target_vals[2]]
shape_waveform_const["Rs_lower_outer"]["times"] = [0.0]
shape_waveform_const["Rs_lower_outer"]["vals"] = [start_target_vals[3]]

print(shape_waveform_const)
with open("./control_pickles/shape_waveform_const.pkl", "wb") as f:
    pickle.dump(shape_waveform_const, f)

### Ip 
Now we repeat the process for Ip control. As with the shape control we'll set the waveform to keep the plasma current constant, starting from the initial equilibrium values.

In [ ]:
# schedule with Ip control off
ip_schedule_1 = {
    0.0: [],
}

# schedule with Ip control on for al time.
ip_schedule_2 = {0.0: ["Ip"]}

with open("./control_pickles/ip_schedule_off.pkl", "wb") as f:
    pickle.dump(ip_schedule_1, f)

with open("./control_pickles/ip_schedule_on.pkl", "wb") as f:
    pickle.dump(ip_schedule_2, f)

In [ ]:
# Ip waveform
ip_start = profiles.Ip
ip_waveform_const = {"Ip": {"times": [0.0], "vals": [ip_start]}}

with open("./control_pickles/ip_waveform_const.pkl", "wb") as f:
    pickle.dump(ip_waveform_const, f)

We also need a few other dictionaries we need to create. 
- CoilPerturbation waveform. These are coil currents which get added to requested control oens, but for now we'll just set this to zero. The simulator still require this as an input so we'll create the necessary empty waveforms now.
- ip_control_parameters : This describes the control sequence for the Ip, and contains values for feedforward Vloop, blends (for mixing feedfoward and feedback Vloop), Kp (proportional gains). For simplicity we'll again use a single constant value for these parameters. In general there'll be multiple values.

In [ ]:
active_coils = eq.tokamak.coils_list[: eq.tokamak.n_active_coils]
print(active_coils)
coil_pert_schedule = {
    0.0: active_coils,
}
with open("./control_pickles/coil_pert_schedule.pkl", "wb") as f:
    pickle.dump(coil_pert_schedule, f)

timestamps = np.arange(0, 1, 0.05)
coil_pert_waveform = {
    coil: {"times": timestamps, "vals": np.zeros_like(timestamps)}
    for coil in active_coils
}

with open("./control_pickles/coil_pert_waveform.pkl", "wb") as f:
    pickle.dump(coil_pert_waveform, f)

In [ ]:
# ip control parameters
ip_copntrol_params = {
    "Kp": {
        0: 1,
    },
    "Vloop": {
        0: 10050,
    },
    "blend": {
        0: 0.5,
    },
    "vc": np.array(
        [1, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.0]
    ),
}

with open("./control_pickles/ip_control_params.pkl", "wb") as f:
    pickle.dump(ip_copntrol_params, f)

Lastly we need the VC schedule. For simplicity we'll use only one VC for the entire simulation, but in general we'd use more. We'll create a VC built from the start equilibrium. Because we're using the VC class in freegsnke, we need to unpack the elements of this to store in the dictionary.

We can specify the coils and targets to be used in the VC. We'll use the default all active coils and set the targets to be those we are controlling.

In [ ]:
VCH.calculate_VC(
    eq=eq,
    profiles=profiles,
    coils=[
        "Solenoid",
        "PX",
        "D1",
        "D2",
        "D3",
        "Dp",
        "D5",
        "D6",
        "D7",
        "P4",
        "P5",
        "P6",
    ],
    targets=["R_in", "R_out", "Rx_lower", "Rs_lower_outer"],
    targets_options=None,
    non_standard_targets=None,
)

virtual_circuit = VCH.latest_VC
gains_arr = 0.5 * np.ones(4)  # target gains - set to 0.5 for all target for testing.
vc_dict = {
    "shape_matrix": virtual_circuit.shape_matrix,
    "vc_matrix": virtual_circuit.VCs_matrix,
    "targets": ["R_in", "R_out", "Rx_lower", "Rs_lower_outer"],
    "coils": virtual_circuit.coils,
    "time_calc": 0.0,
    "time_stop": 10.0,  # set time stop to much longer than simulation time as we're only using one vc.
    "targets_val": virtual_circuit.targets_val,
    "input_currents": virtual_circuit.eq.tokamak.getCurrents(),
    "input_profile_pars": [],  # input_profile_pars - can be obtained from the profiles object
    "target_gains": gains_arr,
}

vc_schedule_dict = {0: vc_dict}  # single vc stored in dictionary

# save to pickle
with open("./control_pickles/vc_schedule.pkl", "wb") as f:
    pickle.dump(vc_schedule_dict, f)

## Control modules

NB. This part can be skipped on first reading if only interested in running a full simulation using the simulation functions.

In this section we'll look at the internal elements of the control modules, and how they are used to implement the control loops. 

The internals of the control loop are contained in the following modules:
- freegsnke.control_loop.target_scheduler.py
- freegsnke.control_loop.vc_scheduler.py
- freegsnke.control_loop.shape_control.py
- freegsnke.control_loop.ip_control.py
- freegsnke.control_loop.pcs.py

We'll look at each of these in turn. The first two deal with the scheduling, the second two perform individual control tasks, and the last one is combines this all into the overall control system.

The target_scheuler module contains the base scheduling class from which the others inheret. This loads the schedule and waveform dictionaries, and then has methods to interpolate the values of the targets at a given time, and to retrieve the values of the targets at a given time.

The vc_scheduler module contains the class that builds the virtual circuits, and stores the sequence of virtual circuits along with appropriate time stamps. 


In [ ]:
# Imports
from freegsnke.control_loop import shape_scheduling
from freegsnke.control_loop import shape_targets_control
from freegsnke.control_loop import ip_control

Lets create a shape_scheduler, which is part of the vc_scheduler module. This loads the shape schedule, waveform and vc schedule from the pickle files.

In [ ]:
ShapeScheduler = shape_scheduling.ShapeTargetScheduler(
    target_schedule_path="./control_pickles/shape_schedule_on.pkl",
    target_waveform_path="./control_pickles/shape_waveform_const.pkl",
    vc_flag="file",
    vc_schedule_path="./control_pickles/vc_schedule.pkl",
)

# retrieve quantities at time t=0.25
t = 0.25
vc = ShapeScheduler.vc_scheduler.retrieve_vc(t)
print(vc.shape_matrix)

targ_names = ShapeScheduler.retrieve_controlled_targets(t)
print(targ_names)

targ_vals = ShapeScheduler.desired_target_values(t)
print(targ_vals)

Ip control is done using the Solenoid so the control class is called SolenoidScheduler. This takes the schedule and waveform for Ip (treating Ip as the 'target' now) and also the ip_control_parameters (analogus to providng the VC scheule to the shape control class).

In [ ]:
IpScheduler = ip_control.SolenoidScheduler(
    target_schedule_path="./control_pickles/ip_schedule_on.pkl",
    target_waveform_path="./control_pickles/ip_waveform_const.pkl",
    control_params_path="./control_pickles/ip_control_params.pkl",
    solenoid_name="Solenoid",
)

In [ ]:
print(IpScheduler.target_waveform_dict)

## Running a simulation 

To run the simulation there are some parameters that need to be set for the machine such as inductances, resistances, etc. We'll set these now (NB these not necessarily the correct/phsyical values, just for testing). Some of these, such as coil inductance matrix and resistances have values stored in the tokamak object, but they can be prescribed here instead.

In [ ]:
test_config_kwargs = {
    "plasma_resistivity": 1e-6,
    "plasma_inductance": 3.9,
    "plas_sol_inductance": 2.7,
    "Rp": 0.84,
    # "R_vec":..., ## leave blank - use resistances from tokamak machine config.
    "coil_gains": np.diag(np.ones(12)),
    # "coil_gains": np.diag(np.random.rand(12)),
    # "target_gains" : ????,  #these are given in VC schedule???
    # "inductance_matrix": ..., ## leaving blank - use inductances from machine config
}

To run the simulation, currently there is a 'simulate shot' function in the control loop module. This takes the following inputs:
- eq_start : equilibrium object
- profiles_start : profiles object
- stepping : stepping object
- t_start : starting time for simulation
- n_iter : number of iterations to simulate
- config_kwargs : dictionary of configuration parameters (values for resistances, inductances, etc.)    
- control_kwargs : dictionary of configuration filepaths (schedules, waveforms, etc)

The function returns a dictionary containing the history of the simulation.

NB THIS MAY CHANGE IN THE FUTURE.

In [ ]:
control_kwargs_all_on = {
    "ip_schedule": "./control_pickles/ip_schedule_on.pkl",
    "ip_control_params": "./control_pickles/ip_control_params.pkl",
    "ip_waveform": "./control_pickles/ip_waveform_const.pkl",
    "fb_target_waveform": "./control_pickles/shape_waveform_const.pkl",
    "fb_target_schedule": "./control_pickles/shape_schedule_on.pkl",
    "ff_target_waveform": "./control_pickles/shape_waveform_const.pkl",
    "ff_target_schedule": "./control_pickles/shape_schedule_on.pkl",
    "vc_schedule": "./control_pickles/vc_schedule.pkl",
    "vc_flag": "file",
    "coil_pert_schedule": "./control_pickles/coil_pert_schedule.pkl",
    "coil_pert_waveform": "./control_pickles/coil_pert_waveform.pkl",
}

control_kwargs_off = {
    "ip_schedule": "./control_pickles/ip_schedule_off.pkl",
    "ip_control_params": "./control_pickles/ip_control_params.pkl",
    "ip_waveform": "./control_pickles/ip_waveform_const.pkl",
    "fb_target_waveform": "./control_pickles/shape_waveform_const.pkl",
    "fb_target_schedule": "./control_pickles/shape_schedule_off.pkl",
    "ff_target_waveform": "./control_pickles/shape_waveform_const.pkl",
    "ff_target_schedule": "./control_pickles/shape_schedule_off.pkl",
    "vc_schedule": "./control_pickles/vc_schedule.pkl",
    "vc_flag": "file",
    "coil_pert_schedule": "./control_pickles/coil_pert_schedule.pkl",
    "coil_pert_waveform": "./control_pickles/coil_pert_waveform.pkl",
}

control_kwargs_shape_on = {
    "ip_schedule": "./control_pickles/ip_schedule_off.pkl",
    "ip_control_params": "./control_pickles/ip_control_params.pkl",
    "ip_waveform": "./control_pickles/ip_waveform_const.pkl",
    "fb_target_waveform": "./control_pickles/shape_waveform_const.pkl",
    "fb_target_schedule": "./control_pickles/shape_schedule_on.pkl",
    "ff_target_waveform": "./control_pickles/shape_waveform_const.pkl",
    "ff_target_schedule": "./control_pickles/shape_schedule_on.pkl",
    "vc_schedule": "./control_pickles/vc_schedule.pkl",
    "vc_flag": "file",
    "coil_pert_schedule": "./control_pickles/coil_pert_schedule.pkl",
    "coil_pert_waveform": "./control_pickles/coil_pert_waveform.pkl",
}

In [ ]:
with open("./control_pickles/shape_waveform_const.pkl", "rb") as f:
    input_waveforms = pickle.load(f)
    print("waveforms loaded")

print(input_waveforms)

In [ ]:
from freegsnke.control_loop.pcs import simulate_shot

plot_eqi(eq, show=True)
stepping.initialize_from_ICs(eq, profiles)

sim_hist = simulate_shot(
    eq_start=eq,
    profiles_start=profiles,
    stepping=stepping,
    t_start=0.0,
    n_iter=10,
    config_kwargs=test_config_kwargs,
    control_kwargs=control_kwargs_off,
)

Now we can run the simulation.

This simulation function returns a dictionary containing the history of the simulation, so we can now plot the results. We will plot the input waveforms and the history of observed target/Ip values to see how well the control loops are working.

In [ ]:
with open("./control_pickles/shape_waveform_const.pkl", "rb") as f:
    shape_waveform_const = pickle.load(f)
print(shape_waveform_const)

with open("./control_pickles/ip_waveform_const.pkl", "rb") as f:
    ipwave = pickle.load(f)

print(ipwave)

Plotting the evolution of plasma 

In [ ]:
# Plot evolution of tracked values and compare between linear and non-linear evolution
from pprint import pprint


def plot_evolution(sim_hist, input_waveforms=None):
    """Plots the evolution of tracked values and compares between linear and non-linear evolution.

    Parameters
    ----------
    sim_hist : dict
        Dictionary containing the simulation history.

    Returns
    -------
    None
    """
    # load simuilatoin history arrays for plotting
    history_times = sim_hist["times"]
    history_voltages = sim_hist["voltages"]
    history_ip = sim_hist["Ip"]
    history_plasma_resistivity = sim_hist["plasma_resistivity"]
    history_jz = sim_hist["jz"]
    # history_width = sim_hist["width"]
    history_o_points = sim_hist["o_points"]
    # history_elongation = sim_hist["elongation"]
    history_currents = sim_hist["full_currents"]
    history_Rin = sim_hist["R_in"]
    history_Rout = sim_hist["R_out"]
    history_xpoints = sim_hist["xpoints"]

    # add end points for plotting flat continuation of input waveforms
    end_time = history_times[-1]
    input_wave_aux = deepcopy(input_waveforms)
    input_wave_aux["R_in"]["times"] = np.append(
        input_wave_aux["R_in"]["times"], end_time
    )
    input_wave_aux["R_in"]["vals"] = np.append(
        input_wave_aux["R_in"]["vals"], input_waveforms["R_in"]["vals"][-1]
    )
    input_wave_aux["R_out"]["times"] = np.append(
        input_wave_aux["R_out"]["times"], end_time
    )
    input_wave_aux["R_out"]["vals"] = np.append(
        input_wave_aux["R_out"]["vals"], input_waveforms["R_out"]["vals"][-1]
    )
    input_wave_aux["Rx_lower"]["times"] = np.append(
        input_wave_aux["Rx_lower"]["times"], end_time
    )
    input_wave_aux["Rx_lower"]["vals"] = np.append(
        input_wave_aux["Rx_lower"]["vals"], input_waveforms["Rx_lower"]["vals"][-1]
    )
    input_wave_aux["Ip"]["times"] = np.append(input_wave_aux["Ip"]["times"], end_time)
    input_wave_aux["Ip"]["vals"] = np.append(
        input_wave_aux["Ip"]["vals"], input_waveforms["Ip"]["vals"][-1]
    )

    pprint(input_wave_aux)

    # create figure and axes - 2x3 grid.
    fig, axs = plt.subplots(2, 3, figsize=(10, 5), dpi=80, constrained_layout=True)
    axs_flat = axs.flat

    axs_flat[0].plot(history_times, history_o_points[:, 0], "k+", label="linear")
    # axs_flat[0].plot(history_times_nl, history_o_points_nl[:, 0], "rx", label="nonlinear")
    axs_flat[0].set_xlabel("Time")
    axs_flat[0].set_ylabel("O-point $R$")
    axs_flat[0].legend()

    # axs_flat[1].plot(history_times, abs(history_o_points[:, 1]), "k+")
    # # axs_flat[1].plot(history_times_nl, abs(history_o_points_nl[:, 1]), "rx")
    # axs_flat[1].set_yscale("log")
    # axs_flat[1].set_xlabel("Time")
    # axs_flat[1].set_ylabel("abs( O-point $Z$ )")

    axs_flat[1].plot(history_times, history_xpoints[:, 1], "k+")
    # axs_flat[1].plot(input_waveforms["Zx"], history_o_points_nl[:, 2], "rx")
    axs_flat[1].set_xlabel("Time")
    axs_flat[1].set_ylabel("Zx")

    axs_flat[2].plot(history_times, history_ip, "k+")
    # axs_flat[2].plot(
    #     history_times, history_currents[:, -1] * stepping.plasma_norm_factor, "k+"
    # )
    axs_flat[2].plot(
        input_wave_aux["Ip"]["times"],
        input_wave_aux["Ip"]["vals"],
        "rx",
        linestyle="--",
    )
    axs_flat[2].set_xlabel("Time")
    axs_flat[2].set_ylabel("Plasma current Ip")

    axs_flat[3].plot(history_times, history_xpoints[:, 0], "k+")
    axs_flat[3].plot(
        input_wave_aux["Rx_lower"]["times"],
        input_wave_aux["Rx_lower"]["vals"],
        "rx",
        linestyle="--",
    )
    axs_flat[3].set_xlabel("Time")
    axs_flat[3].set_ylabel("Rx")

    axs_flat[4].plot(history_times, history_Rin, "k+")
    axs_flat[4].plot(
        input_wave_aux["R_in"]["times"],
        input_wave_aux["R_in"]["vals"],
        "rx",
        linestyle="--",
    )
    axs_flat[4].set_xlabel("Time")
    axs_flat[4].set_ylabel("Rin")

    axs_flat[5].plot(history_times, history_Rout, "k+")
    axs_flat[5].plot(
        input_wave_aux["R_out"]["times"],
        input_wave_aux["R_out"]["vals"],
        "rx",
        linestyle="--",
    )
    axs_flat[5].set_xlabel("Time")
    axs_flat[5].set_ylabel("Rout")

### Testing with Shape control

In [ ]:
plot_eqi(eq, show=True)
stepping.initialize_from_ICs(eq, profiles)

sim_hist_shape_on, input_waveforms_on = simulate_shot(
    eq_start=eq,
    profiles_start=profiles,
    stepping=stepping,
    t_start=0.0,
    n_iter=20,
    config_kwargs=test_config_kwargs,
    control_kwargs=control_kwargs_shape_on,
)

plot_evolution(sim_hist_shape_on, input_waveforms_on)

## Run a validation test

